# Lecture 1 — Losses, likelihood, and the optimization problem

**Week 2 · Day 1 · 45 min**

> **Headline.** A model is a loss. A loss is a negative log-likelihood.

Today we set up the object every other day of this week operates on, and we build the
one tool that tells us whether we got the mathematics right: a gradient check.

**By the end of this lecture you can:**

1. write the optimality conditions for a smooth problem and say when they are sufficient;
2. derive squared error, logistic loss and the Poisson loss as negative log-likelihoods;
3. write any of them as one GLM formula and differentiate it;
4. read regularization as a prior;
5. explain why the *scale of your data* decides how hard the optimization will be;
6. verify a gradient with finite differences, and know why the error stops improving.

**You implement this afternoon:** `numerical_gradient`, `check_gradient`,
`SquaredError`, `LogisticNLL`, `GLMLoss`, `Quadratic`, `Rosenbrock`.

### Pacing

Target **44 min** of core material, hard cap **45 min**. This is the longest lecture of
the week because §1 recalls the calculus everything else stands on — **do not rush it.**
Sections marked *(cut first)* are re-derived later anyway.

| § | Section | min |
|---|---|---|
| 1 | What we need from calculus | 12 |
| 2 | The problem we will solve | 2 |
| 3 | Where does a loss come from? | 10 |
| 4 | Two examples you can check on paper | 4 |
| 5 | A trap: computing the logistic loss naively | 3 |
| 6 | Regularization is a prior  *(cut first)* | 4 |
| 7 | Conditioning: your data decides your difficulty | 5 |
| 8 | Checking a gradient | 6 |
| 9 | Today's labs | 2 |
| | **total** | **48** |
| | **core only** | **44** |

> If your group is already fluent in multivariate calculus, §1 can be set as pre-reading
> and the lecture starts at §2.

---

## 1. What we need from calculus  *(12 min)*

> **Nothing in this week is harder than what is in this section.** If you are comfortable
> here, you are comfortable all week. Read it slowly; everything later is built on it.

We need four things: the **derivative**, the **gradient**, the **Hessian**, and the idea
of a **local model**. Let us recall each one, and — more importantly — *see* it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

# One consistent look for every figure in the lecture.
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.figsize": (11, 3.8),   # these figures use the default
    "font.size": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# If this import fails:   pip install matplotlib

rng = np.random.default_rng(0)

### 1.1 One variable: the derivative

The derivative $f'(x)$ is the **slope of the tangent** at $x$ — how fast $f$ changes if
you nudge $x$:

$$f'(x) = \lim_{h \to 0}\frac{f(x+h) - f(x)}{h}$$

Two readings, both used constantly this week:

- **positive slope** → $f$ increases as you move right, so to *decrease* $f$ you go left;
- **zero slope** → the tangent is flat. The function is momentarily not changing.

A point where $f'(x) = 0$ is called a **stationary** (or critical) point. Careful: flat
does not mean *lowest*. It could be a bottom, a top, or a flat spot on the way past.

In [ ]:
def f1(x):
    return 0.1 * x**4 - 0.5 * x**3 - 0.6 * x**2 + 2.0 * x + 3

def df1(x):
    return 0.4 * x**3 - 1.5 * x**2 - 1.2 * x + 2.0

def d2f1(x):
    return 1.2 * x**2 - 3.0 * x - 1.2

x = np.linspace(-2.2, 4.6, 400)
roots = np.sort(np.roots([0.4, -1.5, -1.2, 2.0]).real)      # where f'(x) = 0
lowest = roots[np.argmin(f1(roots))]                        # the deepest of them

fig, ax = plt.subplots(1, 2)

ax[0].plot(x, f1(x), lw=2)
for r in roots:
    if d2f1(r) < 0:
        kind, dy, col = "local max", 12, "tab:orange"          # curves downward
    elif np.isclose(r, lowest):
        kind, dy, col = "global min", -22, "tab:red"
    else:
        kind, dy, col = "local min", -22, "tab:green"
    ax[0].plot(r, f1(r), "o", ms=9, color=col)
    ax[0].annotate(f"{kind}\nf''={d2f1(r):+.1f}", (r, f1(r)), textcoords="offset points",
                   xytext=(0, dy), ha="center", fontsize=9, color=col)
ax[0].set_title("f(x): every marked point has f'(x) = 0")
ax[0].set_xlabel("x"); ax[0].set_ylim(-7.5, 6.5); ax[0].grid(alpha=.3)

ax[1].axhline(0, color="k", lw=.8)
ax[1].plot(x, df1(x), lw=2, color="tab:purple")
for r in roots:
    col = "tab:orange" if d2f1(r) < 0 else ("tab:red" if np.isclose(r, lowest) else "tab:green")
    ax[1].plot(r, 0, "o", ms=9, color=col)
ax[1].set_title("f'(x): the SAME points are where it crosses zero")
ax[1].set_xlabel("x"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

print("stationary points:", np.round(roots, 4))
print("f'' there       :", np.round(d2f1(roots), 4), " <- the sign is what tells min from max")

**Figure 1 — a zero derivative is not enough.**

Look at the two panels together. **Three different kinds of point — one local minimum,
one local maximum, one global minimum — and the derivative cannot tell them apart.** All
it says is "zero".

That is the single most important limitation to carry through the week: every algorithm
we write looks for $\nabla f = 0$, so every algorithm we write can in principle stop at
the wrong kind of point. We will watch exactly that happen on day 4.

To tell them apart we need the **second** derivative — the curvature:

- $f''(x) > 0$: the curve bends **upward** (like $\smile$) → a minimum;
- $f''(x) < 0$: it bends **downward** (like $\frown$) → a maximum;
- $f''(x) = 0$: inconclusive.

### Local versus global

- a **local** minimum is lowest *in some neighbourhood around it*;
- a **global** minimum is lowest **everywhere**.

Derivatives are purely local — they see an infinitesimal neighbourhood and nothing else.
So no method in this course can certify a global minimum. There is one important
exception, in §1.5.

### 1.2 Several variables: partial derivatives and the gradient

Our unknown is not a number, it is a **vector of parameters** $w \in \mathbb{R}^n$
(the coefficients of a model). So $f$ takes a vector and returns a number.

A **partial derivative** $\dfrac{\partial f}{\partial x_i}$ is the ordinary derivative in
coordinate $i$, **holding every other coordinate fixed**: walk parallel to axis $i$ and
measure the slope.

Stack them all into a vector — that is the **gradient**:

$$\nabla f(x) \;=\; \begin{pmatrix} \partial f/\partial x_1 \\ \vdots \\ \partial f/\partial x_n \end{pmatrix}$$

Two facts about $\nabla f$, and they are the reason this week exists:

1. **It points in the direction of steepest increase.** So $-\nabla f$ is the direction of
   steepest *decrease* — which is the whole idea of gradient descent tomorrow.
2. **It is perpendicular to the level sets** (the contour lines, where $f$ is constant).
   That makes sense: moving *along* a contour does not change $f$, so the direction of
   maximal change must be at right angles to it.

**Example.** For $f(x_1,x_2) = x_1^2 + 5x_2^2$:
$\dfrac{\partial f}{\partial x_1} = 2x_1$, $\dfrac{\partial f}{\partial x_2} = 10x_2$,
so $\nabla f = (2x_1,\; 10x_2)$. At $(1,1)$ that is $(2, 10)$ — mostly pointing along
$x_2$, because $f$ climbs five times faster in that direction.

In [ ]:
def f2(X1, X2):
    return X1**2 + 5 * X2**2

g1, g2 = np.meshgrid(np.linspace(-3, 3, 300), np.linspace(-1.6, 1.6, 300))
Z = f2(g1, g2)

fig, ax = plt.subplots(1, 2)

cs = ax[0].contour(g1, g2, Z, levels=[.5, 2, 5, 10, 18, 28], colors="tab:blue")
ax[0].clabel(cs, inline=True, fontsize=8)
ax[0].plot(0, 0, "r*", ms=16)
ax[0].set_title("level sets of  f = x1² + 5·x2²"); ax[0].set_aspect("equal")

ax[1].contour(g1, g2, Z, levels=[.5, 2, 5, 10, 18, 28], colors="lightgray")
px, py = np.meshgrid(np.linspace(-2.5, 2.5, 9), np.linspace(-1.3, 1.3, 7))
ax[1].quiver(px, py, -2 * px, -10 * py, color="tab:red", alpha=.8)
ax[1].plot(0, 0, "r*", ms=16)
ax[1].set_title("the arrows are  −∇f  (downhill, ⊥ to the contours)")
ax[1].set_aspect("equal")
plt.tight_layout(); plt.show()

print("∇f at (1, 1) =", np.array([2 * 1.0, 10 * 1.0]), " -> climbs 5x faster along x2")

**Figure 2 — the gradient field of a stretched bowl.**

Notice the contours are **ellipses, not circles**, because the two coordinates are scaled
differently ($1$ against $5$). And notice the arrows do **not** point straight at the
star: they point across the valley rather than along it.

That mismatch is the central difficulty of the whole week. It has a name — **conditioning**
— and we return to it in §7 with real data, and tomorrow with a formula for exactly how
many iterations it costs you.

### 1.3 Second derivatives: the Hessian

In one variable, curvature was one number $f''$. In $n$ variables we need **every pair**:
how does the slope in direction $i$ change as you move in direction $j$? That is an
$n \times n$ matrix, the **Hessian**:

$$\nabla^2 f(x) \;=\; H \;=\; \begin{pmatrix}
\frac{\partial^2 f}{\partial x_1^2} & \cdots & \frac{\partial^2 f}{\partial x_1 \partial x_n}\\
\vdots & \ddots & \vdots\\
\frac{\partial^2 f}{\partial x_n \partial x_1} & \cdots & \frac{\partial^2 f}{\partial x_n^2}
\end{pmatrix}$$

It is **symmetric** for the functions we use, because $\partial^2 f/\partial x_i \partial x_j$
does not depend on the order of differentiation.

For $f = x_1^2 + 5x_2^2$: $\nabla f = (2x_1, 10x_2)$, so differentiating again gives
$H = \begin{pmatrix} 2 & 0\\ 0 & 10\end{pmatrix}$ — constant, because $f$ is quadratic.

### Reading the Hessian: eigenvalues are curvatures

The **eigenvalues** of $H$ are the curvatures along the special directions (the
eigenvectors) where the surface bends purely up or purely down. That gives us the
multi-dimensional version of the $f'' > 0$ test:

| eigenvalues of $H$ | name | shape | the point is |
|---|---|---|---|
| all $> 0$ | positive definite | a bowl $\smile$ | a **minimum** |
| all $< 0$ | negative definite | a dome $\frown$ | a **maximum** |
| mixed signs | indefinite | a **saddle** | neither |
| all $\ge 0$, some $= 0$ | positive semidefinite | a flat-bottomed valley | inconclusive |

A **saddle** is the genuinely new thing in more than one dimension: a point that is a
minimum along one direction and a maximum along another — like the middle of a horse's
saddle, or a mountain pass. It is flat ($\nabla f = 0$) but it is not a minimum.

In [ ]:
fig = plt.figure(figsize=(12, 4.0))
g1, g2 = np.meshgrid(np.linspace(-2, 2, 120), np.linspace(-2, 2, 120))

cases = [
    ("bowl:  x1² + x2²",   g1**2 + g2**2,  np.diag([2.0, 2.0]),   "MINIMUM"),
    ("saddle:  x1² − x2²", g1**2 - g2**2,  np.diag([2.0, -2.0]),  "SADDLE"),
    ("dome:  −x1² − x2²",  -g1**2 - g2**2, np.diag([-2.0, -2.0]), "MAXIMUM"),
]

for k, (title, Z, H, verdict) in enumerate(cases, 1):
    ax = fig.add_subplot(1, 3, k, projection="3d")
    ax.plot_surface(g1, g2, Z, cmap="viridis", alpha=.6, linewidth=0, antialiased=True)
    # The stationary point, drawn on top of the surface so it stays visible.
    ax.plot([0], [0], [0], "o", color="red", ms=11, zorder=10)
    ax.plot([0, 0], [0, 0], [Z.min(), 0], "--", color="red", lw=1.2, alpha=.8, zorder=9)
    ax.set_title(f"{title}\neigenvalues {np.linalg.eigvalsh(H)}  ->  {verdict}", fontsize=9)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
plt.tight_layout(); plt.show()

print("At the red point all three have gradient (0, 0) -- all three are 'flat'.")
print("Only the eigenvalues of the Hessian tell minimum from saddle from maximum.")

**Figure 3 — the three things a stationary point can be.**

Three quadratic surfaces, and the red marker is the stationary point of each — the point
where $\nabla f = 0$. All three satisfy that condition, and they could hardly be more
different: the first is the minimum we want, the second falls away in one direction while
rising in another, and the third is a maximum.

The Hessian is what tells them apart, and the next cell turns that into four lines of
code. Read the verdicts in the titles against the eigenvalues it prints: **both positive**
is a bowl, **mixed signs** is a saddle, **both negative** is a dome.

> A saddle is the case worth fearing. It is not rare in high dimensions — it is the
> *typical* stationary point, because a minimum needs every one of $n$ eigenvalues to be
> positive, and a saddle needs only one to be negative.

In [ ]:
def classify(H):
    """The test you will use all week, in four lines."""
    eig = np.linalg.eigvalsh(H)
    if np.all(eig > 0):  kind = "MINIMUM  (positive definite)"
    elif np.all(eig < 0): kind = "MAXIMUM  (negative definite)"
    elif np.all(eig >= 0): kind = "inconclusive (positive semidefinite)"
    else: kind = "SADDLE   (indefinite)"
    return f"eigenvalues {np.round(eig, 3)}  ->  {kind}"

for name, H in [("x1² + 5x2²",  np.diag([2.0, 10.0])),
                ("x1² − x2²",   np.diag([2.0, -2.0])),
                ("−x1² − x2²",  np.diag([-2.0, -2.0])),
                ("x1²",         np.diag([2.0, 0.0])),
                ("[[1,2],[2,1]]", np.array([[1.0, 2.0], [2.0, 1.0]]))]:
    print(f"{name:>15} : {classify(H)}")

Remember that last one, $\begin{pmatrix}1&2\\2&1\end{pmatrix}$ — eigenvalues $3$ and
$-1$, so a saddle. It comes back on **day 4** as the matrix on which the Cholesky
factorization must fail, and that failure will turn out to be useful information rather
than a bug.

### 1.4 Taylor: building a local model

We cannot minimize a complicated $f$ directly. So we do what all of numerical
optimization does: **replace $f$ near the current point by something simple, minimize
that, move, repeat.** "Something simple" means a Taylor expansion.

In one variable, around $x$:

$$f(x + h) \;\approx\; \underbrace{f(x)}_{\text{value}} + \underbrace{f'(x)\,h}_{\text{slope}} + \underbrace{\tfrac{1}{2}f''(x)\,h^2}_{\text{curvature}}$$

In $n$ variables, with a step $p \in \mathbb{R}^n$, exactly the same three terms:

$$\boxed{\;f(x + p) \;\approx\; f(x) \;+\; \nabla f(x)^\top p \;+\; \tfrac{1}{2}\,p^\top \nabla^2 f(x)\,p\;}$$

Note the shapes: $\nabla f^\top p$ is a number (vector times vector), and
$p^\top H p$ is a number (vector times matrix times vector). Both are scalars, as they
must be, since $f$ returns a number.

**This one formula generates the entire week:**

| keep | model | method | day |
|---|---|---|---|
| value + slope | a **plane** | gradient descent | 2, 3 |
| value + slope + curvature | a **parabola** | Newton | 4 |
| curvature approximated from first derivatives | a cheap parabola | Gauss–Newton, LM | 5 |

Every method differs only in *which Taylor model it trusts, and how far*.

In [ ]:
# The Taylor models of f(x) = exp(x) at x = 0, drawn against the truth.
x = np.linspace(-1.6, 1.6, 300)
plt.figure(figsize=(6, 3.8))
plt.plot(x, np.exp(x), lw=2.5, label="f(x) = exp(x)")
plt.plot(x, 1 + x, "--", lw=2, label="linear model: 1 + x")
plt.plot(x, 1 + x + x**2 / 2, "--", lw=2, label="quadratic: 1 + x + x²/2")
plt.plot(0, 1, "ko", ms=8)
plt.legend(fontsize=9); plt.grid(alpha=.3)
plt.title("a model is only trustworthy NEAR the point")
plt.tight_layout(); plt.show()

for h in [0.1, 0.5, 1.5]:
    true = np.exp(h)
    print(f"h={h:4.1f}  true={true:8.4f}   linear={1+h:8.4f} (err {abs(true-1-h):7.4f})"
          f"   quadratic={1+h+h*h/2:8.4f} (err {abs(true-1-h-h*h/2):7.4f})")

**Figure 4 — the linear and quadratic models, against the truth.**

Two lessons from that table, and both shape the week:

- the quadratic model is **much** more accurate than the linear one — which is why Newton
  converges in a handful of steps where gradient descent needs hundreds;
- but **both degrade as $h$ grows.** A model is local. Trusting it too far is precisely
  how Gauss–Newton diverges on day 5, and controlling how far to trust it is what a line
  search (day 2) and a trust region (day 5) are for.

### 1.5 Convexity: when local is global

One more idea, and it is the one that makes any of this reliable.

$f$ is **convex** if the line segment between any two points on its graph lies *above* the
graph — no bumps, one valley. Equivalently, for a twice-differentiable $f$:

$$f \text{ is convex} \iff \nabla^2 f(x) \succeq 0 \text{ for every } x$$

("$\succeq 0$" means positive semidefinite: every eigenvalue $\ge 0$, curving upward or
flat in every direction, never downward.)

**Why it matters so much:** for a convex $f$, **every stationary point is a global
minimum.** No local traps, no saddles to get stuck in. So $\nabla f = 0$ stops being merely
necessary and becomes genuinely sufficient.

This is why the losses in §3 being convex is not a footnote. It is the reason logistic
regression fits reliably and training a neural network does not.

In [ ]:
x = np.linspace(-3, 3, 400)
convex, nonconvex = x**2, 0.12 * x**4 - 0.9 * x**2 + 0.35 * x

fig, ax = plt.subplots(1, 2)
for a, (y, title) in zip(ax, [(convex, "CONVEX: x²  — one valley"),
                              (nonconvex, "NOT convex — two valleys, a bump between")]):
    a.plot(x, y, lw=2)
    a.plot([-2.3, 2.3], [np.interp(-2.3, x, y), np.interp(2.3, x, y)], "o--",
           color="tab:red", alpha=.7, label="chord")
    a.set_title(title, fontsize=10); a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout(); plt.show()

print("Left : the chord is never below the curve -> convex -> any flat point is THE minimum.")
print("Right: the chord dips below -> not convex -> a flat point may be a local trap.")

**Figure 5 — convexity: when a local minimum is the global one.**

The definition is the picture: $f$ is **convex** when the straight chord between any two
points on the graph never dips below the graph itself.

*Left:* $x^2$. Every chord lies on or above the curve. There is one valley, so a point
with $f'(x) = 0$ has nowhere else to be — **stationary implies global minimum**, and the
gap between the necessary condition of §1.1 and the thing we actually want closes
completely.

*Right:* a quartic with two valleys and a bump between them. The drawn chord passes
*below* the curve, so this $f$ is not convex — and now $f'(x) = 0$ happens at three
points: two minima of different depths, and the bump. An optimizer that lands in the
shallower valley has satisfied every condition we can check locally, and is still in the
wrong place.

> **This is why convexity is the dividing line of the field.** On a convex problem "find a
> stationary point" and "solve the problem" are the same task. Off it they are not, and no
> method in this course can tell you which valley you are in — a point that returns on
> day 5 with a concrete bill attached.

The good news for this week: the GLM losses of §3 **are** convex, so days 2–4 are working
on the left-hand picture. Day 5's curve fitting is not, and day 5 says so loudly.

### Summary of §1 — the vocabulary for the rest of the week

| symbol | name | shape | says |
|---|---|---|---|
| $f(x)$ | the objective | scalar | what we minimize |
| $\nabla f(x)$ | gradient | vector, $n$ | steepest-increase direction; $0$ at a flat point |
| $\nabla^2 f(x)$ | Hessian | matrix, $n \times n$ | curvature; its eigenvalues classify the point |
| $\kappa$ | condition number | scalar | how elongated the valley is — how hard the problem is |

And the three questions we ask about any point:

1. Is $\nabla f = 0$? *(flat — a candidate)*
2. Is $\nabla^2 f \succ 0$? *(a bowl — so genuinely a minimum, not a saddle)*
3. Is $f$ convex? *(if yes, that minimum is the global one)*

> **If only one thing survives this lecture:** the gradient tells you *which way is down*,
> the Hessian tells you *what shape you are standing in*, and Taylor tells you *how far
> you may trust either*.

---

## 2. The problem we will solve  *(2 min)*

With that vocabulary, the whole week is one line:

$$\min_{x \in \mathbb{R}^n} f(x)$$

We look for a point where we cannot improve locally — so, by §1, a point where
$\nabla f(x) = 0$, and where $\nabla^2 f(x) \succ 0$ so that it is a bowl and not a saddle.
If $f$ is convex, that point is the global minimum and we are done.

Everything that follows is about two questions:

1. **Where does $f$ come from?** (§3–§5: it is a negative log-likelihood)
2. **How do we find that point?** (days 2–6: follow $-\nabla f$, and use $\nabla^2 f$ when
   we can afford it)

The only thing still missing is a way to know whether the $\nabla f$ *we coded* is really
the gradient of the $f$ *we coded*. That is §8, and we build it this afternoon.

The last matrix, $\begin{pmatrix}1&2\\2&1\end{pmatrix}$, has eigenvalues $3$ and $-1$.
It will come back on day 4 as the matrix on which Cholesky must **fail**.

---

## 3. Where does a loss come from?  *(10 min)*

Here is the question that organizes the week. When you fit a model you minimize
something. Where does that something come from — why squared error and not absolute
error, why cross-entropy and not accuracy?

**Answer: the loss is the negative log-likelihood of a probabilistic model of the data.**
You choose a distribution for how $y$ is generated given $x$; the loss follows.

Given data $(x_i, y_i)_{i=1}^n$ assumed independent, the likelihood of parameters $w$ is

$$L(w) = \prod_{i=1}^n p(y_i \mid x_i, w).$$

Products of small numbers underflow and are painful to differentiate, so we take the
logarithm — monotone, so the maximizer does not move — and flip the sign to get a
minimization:

$$\text{loss}(w) = -\frac{1}{n}\sum_{i=1}^n \log p(y_i \mid x_i, w).$$

Maximum likelihood estimation *is* minimizing this. Now let us turn the crank three
times.

### 3.1 Gaussian noise gives squared error

Assume $y_i = x_i^\top w + \varepsilon_i$ with $\varepsilon_i \sim \mathcal{N}(0, \sigma^2)$.
Write $z_i = x_i^\top w$ for the **linear predictor**. Then

$$p(y_i \mid x_i, w) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\!\left(-\frac{(z_i - y_i)^2}{2\sigma^2}\right)$$

$$-\log p(y_i \mid x_i, w) = \frac{(z_i - y_i)^2}{2\sigma^2} + \underbrace{\tfrac{1}{2}\log(2\pi\sigma^2)}_{\text{constant in } w}$$

Drop the constant and set $\sigma = 1$:

$$\boxed{\varphi(z, y) = \tfrac{1}{2}(z - y)^2}$$

**Least squares is the Gaussian maximum-likelihood estimator.** The choice of squared
error is a claim that your noise is Gaussian — which also tells you when it is the wrong
choice. Gaussian tails are extremely thin, so a single far-out point is wildly improbable
under the model, and the fit will distort itself to accommodate it. That is exactly the
outlier sensitivity we fix on day 6 with the Huber loss.

### 3.2 Bernoulli gives the logistic loss

Now $y_i \in \{0, 1\}$, and we model $P(y_i = 1 \mid x_i) = \sigma(z_i)$ where
$\sigma(z) = 1/(1 + e^{-z})$ is the logistic function. Both cases at once:

$$p(y_i \mid x_i, w) = \sigma(z_i)^{y_i}\,(1 - \sigma(z_i))^{1 - y_i}$$

$$-\log p = -y_i \log \sigma(z_i) - (1 - y_i)\log(1 - \sigma(z_i))$$

Using $\log \sigma(z) = -\log(1 + e^{-z})$ and $1 - \sigma(z) = \sigma(-z)$, this collapses to

$$\boxed{\varphi(z, y) = \log(1 + e^{z}) - y\,z}$$

This is cross-entropy, and it is the Bernoulli negative log-likelihood. Its derivative is
remarkably clean:

$$\frac{\partial \varphi}{\partial z} = \sigma(z) - y, \qquad
\frac{\partial^2 \varphi}{\partial z^2} = \sigma(z)\,(1 - \sigma(z))$$

The first derivative is *predicted minus observed*. The second is strictly positive, so
the loss is strictly convex in $z$ — and it is the diagonal weight that will appear in
the Hessian on day 4.

### 3.3 Poisson counts  *(optional — cut first)*

If $y_i$ is a count with $y_i \sim \text{Poisson}(e^{z_i})$ (the log link keeps the rate
positive), then $p(y) = \lambda^y e^{-\lambda}/y!$ with $\lambda = e^{z}$ gives

$$-\log p = e^{z} - y\,z + \log(y!) \quad\Longrightarrow\quad \boxed{\varphi(z, y) = e^{z} - y\,z}$$

You implement this one on **day 6** — and the point of doing it then is that it will cost
you nothing. Gradient descent, Newton and ridge will all work on it the moment the class
exists, because none of them knows what a likelihood is.

### 3.4 One formula for all of them

Notice what just happened. Three different distributions, three different losses, but
each one has the same shape: a function of the **linear predictor** $z_i = x_i^\top w$
and the observation $y_i$. So write the whole family at once:

$$\boxed{\;L(w) \;=\; \frac{1}{n}\sum_{i=1}^n \varphi(x_i^\top w,\; y_i) \;=\; \frac{1}{n}\sum_{i=1}^n \varphi(z_i, y_i)\;}$$

This is the **generalized linear model**. Only $\varphi$ changes between linear
regression, logistic regression and Poisson regression.

Differentiate by the chain rule. With $z = Xw$ where $X$ is $n \times p$:

$$\frac{\partial L}{\partial w} = \frac{1}{n}\sum_i \varphi'(z_i, y_i)\,x_i
\;=\; \boxed{\frac{1}{n} X^\top \varphi'(Xw, y)}$$

One gradient formula for every GLM. This is why `GLMLoss` is **one class** that receives
an `IPointwiseLoss`, rather than three classes that each reimplement $X^\top(\cdot)/n$.
The mathematics says these models differ in exactly one place, so the code should differ
in exactly one place too.

> **Design note.** `GLMLoss` has no `lambda` parameter. Regularization is a *separate*
> object that day 4 wraps around it. A penalty is not part of the likelihood — mixing
> them would be the same modelling error as a code smell.

In [ ]:
def softplus(z):
    """log(1 + exp(z)), computed so it does not overflow. See section 4."""
    return np.maximum(z, 0) + np.log1p(np.exp(-np.abs(z)))


def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


# The three pointwise losses and their first derivatives.
PHI = {
    "squared":  (lambda z, y: 0.5 * (z - y) ** 2,   lambda z, y: z - y),
    "logistic": (lambda z, y: softplus(z) - y * z,  lambda z, y: sigmoid(z) - y),
    "poisson":  (lambda z, y: np.exp(z) - y * z,    lambda z, y: np.exp(z) - y),
}


def glm_value(w, X, y, phi):
    """L(w) = mean of phi(Xw, y). The SAME function for all three models."""
    return float(np.mean(PHI[phi][0](X @ w, y)))


def glm_gradient(w, X, y, phi):
    """grad L(w) = X.T @ phi'(Xw, y) / n. Also the same for all three."""
    return X.T @ PHI[phi][1](X @ w, y) / len(y)

---

## 4. Two examples you can check on paper  *(4 min)*

Numbers you can verify by hand are worth more than numbers you cannot: when a test on
them fails, you know *which term* is wrong. Both of these become tests this afternoon.

### 4.1 Squared error

Take $X = \begin{pmatrix}1\\2\end{pmatrix}$, $y = (2, 4)$, and evaluate at $w = 0$.

Then $z = Xw = (0, 0)$, so

$$L(0) = \frac{1}{2}\left[\tfrac12(0-2)^2 + \tfrac12(0-4)^2\right] = \frac{2 + 8}{2} = \boxed{5}$$

$$\nabla L(0) = \frac{1}{n}X^\top(z - y) = \frac{1}{2}\begin{pmatrix}1 & 2\end{pmatrix}\begin{pmatrix}-2\\-4\end{pmatrix} = \frac{-2 - 8}{2} = \boxed{-5}$$

### 4.2 Logistic

Same $X$, now $y = (0, 1)$, still at $w = 0$. Then $z = (0,0)$ and $\sigma(0) = 1/2$:

$$L(0) = \frac{1}{2}\left[\log 2 + \log 2\right] = \boxed{\log 2 \approx 0.6931}$$

$$\nabla L(0) = \frac{1}{2}\begin{pmatrix}1 & 2\end{pmatrix}\begin{pmatrix}0.5 - 0\\0.5 - 1\end{pmatrix} = \frac{0.5 - 1}{2} = \boxed{-0.25}$$

In [ ]:
X = np.array([[1.0], [2.0]])
w0 = np.zeros(1)

print("squared error, y = (2, 4), w = 0")
print("  value   ", glm_value(w0, X, np.array([2.0, 4.0]), "squared"), "  (hand: 5)")
print("  gradient", glm_gradient(w0, X, np.array([2.0, 4.0]), "squared"), "  (hand: -5)")

print("\nlogistic, y = (0, 1), w = 0")
print("  value   ", glm_value(w0, X, np.array([0.0, 1.0]), "logistic"), f"  (hand: log 2 = {np.log(2):.6f})")
print("  gradient", glm_gradient(w0, X, np.array([0.0, 1.0]), "logistic"), "  (hand: -0.25)")

---

## 5. A trap: computing the logistic loss naively  *(3 min)*

$\varphi(z,y) = \log(1 + e^{z}) - yz$ is mathematically fine and numerically a
minefield. For $z = 1000$, $e^{z}$ overflows to `inf`, and `log(inf)` is `inf`. But the
*true* value is about $1000$ — for large $z$, $\log(1 + e^z) \approx z$.

The stable formulation, for any sign of $z$:

$$\log(1 + e^{z}) = \max(z, 0) + \log\!\left(1 + e^{-|z|}\right)$$

The exponential now has a non-positive argument, so it is at most $1$ and never
overflows. This matters in practice: early in training, before $w$ has settled, linear
predictors of a few hundred are routine — and one `inf` poisons the whole gradient.

In [ ]:
z = np.array([1000.0])

with np.errstate(over="ignore"):
    naive = np.log(1 + np.exp(z))

print("naive  log(1 + exp(1000)) =", naive[0])
print("stable softplus(1000)     =", softplus(z)[0], "  <- correct")

# And it still agrees with the naive version where the naive version works.
z = np.linspace(-20, 20, 9)
print("\nmax disagreement on a safe range:", np.max(np.abs(softplus(z) - np.log(1 + np.exp(z)))))

---

## 6. Regularization is a prior  *(4 min — cut first)*

Add a penalty to the loss:

$$\min_w \; \underbrace{\frac{1}{n}\sum_i \varphi(z_i, y_i)}_{\text{fit}} \;+\; \underbrace{r(w)}_{\text{penalty}}$$

Where does $r$ come from? The same place as the loss. Put a **prior** $p(w)$ on the
parameters and maximize the posterior instead of the likelihood (*maximum a posteriori*):

$$\arg\max_w \; p(w \mid \text{data}) = \arg\max_w \; p(\text{data} \mid w)\,p(w)
\;\Longrightarrow\; \arg\min_w \; \big[-\log p(\text{data} \mid w) - \log p(w)\big]$$

The penalty is just $-\log p(w)$:

| Prior on $w$ | $-\log p(w)$ | Penalty | Name |
|---|---|---|---|
| Gaussian $\mathcal{N}(0, \tau^2 I)$ | $\tfrac{1}{2\tau^2}\|w\|_2^2$ | $\tfrac{\lambda}{2}\|w\|_2^2$ | **ridge** (day 4) |
| Laplace $\propto e^{-\|w\|_1/b}$ | $\tfrac{1}{b}\|w\|_1$ | $\lambda\|w\|_1$ | **lasso** (day 6) |
| Flat | constant | $0$ | plain MLE |

Both priors say "coefficients are probably small". They differ in *shape*: the Laplace
density has a spike at zero and heavier tails, so it prefers solutions where most
coefficients are exactly zero and a few are large. The Gaussian, smooth at zero, never
produces an exact zero.

That difference is the whole of day 6 — and it will show up as the difference between a
penalty that is differentiable at $0$ and one that is not.

---

## 7. Conditioning: your data decides your difficulty  *(5 min)*

Here is a fact worth internalizing before you write a single optimizer. **A large part of
optimization difficulty is created by the data, not by the algorithm.**

For least squares the Hessian is $A = X^\top X / n$. Define the **condition number**

$$\kappa(A) = \frac{\lambda_{\max}(A)}{\lambda_{\min}(A)} \;\ge\; 1.$$

Geometrically $\kappa$ is the elongation of the level sets of $f$: $\kappa = 1$ means
circular contours, large $\kappa$ means a long thin valley. Tomorrow we prove that
gradient descent converges at rate $\frac{\kappa - 1}{\kappa + 1}$ per step — so
$\kappa = 10^6$ means essentially no progress.

And where does a large $\kappa$ come from? Often from *nothing but the units of your
features*. One column in metres and another in micrometres, and $\kappa$ explodes with no
statistical content whatsoever.

In [ ]:
# Two mildly correlated, equally informative features.
# The only difference between them: the second is recorded in tiny units.
n = 500
f1 = rng.normal(size=n)
f2 = 0.3 * f1 + rng.normal(size=n)
X_raw = np.column_stack([f1, 1e-3 * f2])

def kappa(X):
    return np.linalg.cond(X.T @ X / len(X))

def standardize(X):
    return (X - X.mean(axis=0)) / X.std(axis=0)

print(f"kappa, raw units    : {kappa(X_raw):12.1f}")
print(f"kappa, standardized : {kappa(standardize(X_raw)):12.1f}")
print("\nSame information, same model. A change of units, and the problem got"
      f" {kappa(X_raw) / kappa(standardize(X_raw)):.0f}x harder.")

So: **standardize your features.** It is not a statistical nicety, it is an optimization
decision, and it is free. In this afternoon's Lab 3 you will measure $\kappa$ on the raw
California housing data and watch it collapse after standardization.

There are two ways to respond to bad conditioning, and this week covers both:

- **fix the data** — standardize (today);
- **fix the algorithm** — use curvature (Newton, day 4) or adapt per coordinate
  (Adam, day 3).

---

## 8. Checking a gradient  *(6 min)*

You are about to write dozens of analytic gradients. Some of them will be wrong. A wrong
gradient does not raise an exception — it silently converges to the wrong place, or
mysteriously fails to converge, and you lose an hour. So we verify every one.

**Forward difference**, from the Taylor expansion $f(x+h) = f(x) + hf'(x) + \tfrac{h^2}{2}f''$:

$$f'(x) \approx \frac{f(x+h) - f(x)}{h} + O(h)$$

**Central difference** — subtract the expansions at $\pm h$ and the quadratic terms cancel:

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h} + O(h^2)$$

Central costs one extra evaluation per coordinate and buys a whole order of accuracy. Use it.

### Why you cannot just take $h \to 0$

Two errors fight each other:

- **truncation**, from stopping the Taylor series: $O(h^2)$ for central differences — shrinks with $h$;
- **rounding**, from subtracting two nearly equal floats and dividing by something tiny:
  roughly $O(\varepsilon_{\text{mach}}/h)$ — *grows* as $h$ shrinks.

Their sum is U-shaped. Minimizing $h^2 + \varepsilon/h$ gives $h \sim \varepsilon^{1/3} \approx 6\times10^{-6}$
for float64, with a best achievable relative error near $10^{-11}$.

You will never do better than about 6 correct digits from finite differences. That is the
reason your Week 1 autodiff module is worth having: it is exact to machine precision, and
this afternoon it serves as a **second, independent oracle** on every gradient you write.

In [ ]:
def numerical_gradient(f, x, h=1e-6):
    """Central differences, one coordinate at a time."""
    g = np.zeros_like(x)
    for i in range(len(x)):
        step = np.zeros_like(x)
        step[i] = h
        g[i] = (f(x + step) - f(x - step)) / (2 * h)
    return g


# The U-curve: sweep h and watch truncation give way to rounding.
f = lambda x: float(np.sum(np.exp(x) + x ** 3))
grad_exact = lambda x: np.exp(x) + 3 * x ** 2
x = np.array([0.7, -1.3])

print(f"{'h':>10} {'relative error':>18}")
for h in [1e-1, 1e-2, 1e-3, 1e-4, 1e-5, 1e-6, 1e-8, 1e-10, 1e-12, 1e-14]:
    err = np.linalg.norm(numerical_gradient(f, x, h) - grad_exact(x)) / np.linalg.norm(grad_exact(x))
    print(f"{h:10.0e} {err:18.3e}")

Read the two arms of that table. Down to about $10^{-6}$ the error falls by ~100 per
decade — that is the $O(h^2)$ truncation term. Past the minimum it *climbs* again — that
is rounding. The bottom of the U is the best finite differences can do.

This is what `check_gradient` will wrap: compare analytic against numerical at a sensible
$h$, and report the relative error

$$\frac{\|g_{\text{analytic}} - g_{\text{numerical}}\|}{\max(1, \|g_{\text{numerical}}\|)} \;<\; 10^{-5}.$$

In [ ]:
# A gradient check must be able to FAIL, or it is checking nothing.
for name, candidate in [("correct", grad_exact), ("wrong (3x^2 -> 2x^2)", lambda x: np.exp(x) + 2 * x ** 2)]:
    gn = numerical_gradient(f, x)
    rel = np.linalg.norm(candidate(x) - gn) / max(1.0, np.linalg.norm(gn))
    print(f"{name:24s} relative error = {rel:.3e}   {'PASS' if rel < 1e-5 else 'FAIL'}")

---

## 9. Today's labs  *(2 min)*

| Lab | What | Why |
|---|---|---|
| 0 | numpy ramp (25 min) | broadcasting, `@`, reductions, `np.linalg` — everything after this assumes fluency |
| 1 | `numerical_gradient`, `numerical_jacobian`, `check_gradient` | the verification tool, plus your Week 1 autodiff as a second oracle |
| 2 | `SquaredError`, `LogisticNLL`, `GLMLoss`, `Quadratic`, `Rosenbrock` | one GLM class for every likelihood |
| 3 | conditioning on real data | $\kappa$ before and after standardization |

**Two questions to be able to answer at the debrief:**

1. Why is squared error the *Gaussian* maximum-likelihood estimator — and what would you
   use instead if you did not believe the noise was Gaussian?
2. Why does rescaling a feature change $\kappa$, when it cannot possibly change what the
   model is able to predict?

### Where this is going

| Day | Question it answers |
|---|---|
| 2 | How do we actually descend? |
| 3 | What if $n$ is too large to touch every sample? |
| 4 | What if we use the curvature $\nabla^2 f$? |
| 5 | What if the loss is a sum of squared residuals? |
| 6 | What if the penalty is not differentiable? |

Every one of them minimizes the object we built today.